# Clase 2: tablas dinámicas, visualización e interpretación de datos

**Duración:** 180 minutos (3 horas reloj)  
**Modalidad:** explicación, demostración, práctica guiada y desafío integrador  
**Herramientas:** Python, Pandas y Matplotlib  
**Dataset:** ventas de una librería escolar, incluido en el notebook

## Objetivos de aprendizaje

Al finalizar la clase, el estudiante será capaz de:

- construir tablas dinámicas mediante `pivot_table()`;
- elaborar tablas de frecuencias con `pd.crosstab()`;
- seleccionar gráficos adecuados para diferentes preguntas;
- crear y personalizar gráficos con Pandas y Matplotlib;
- interpretar tablas y gráficos sin confundir observaciones con opiniones;
- comunicar hallazgos mediante un miniinforme basado en datos.



## 1. Inicio: una decisión basada en datos 
La librería escolar ya corrigió errores y resumió sus ventas. Ahora la administración debe decidir:

> ¿Qué productos debería promocionar en cada sucursal y cómo podemos explicar la decisión de manera clara?

Una lista extensa de números puede ser correcta, pero difícil de interpretar. Las **tablas dinámicas** reorganizan los datos y los **gráficos** hacen visibles las comparaciones, proporciones y tendencias.

### Preguntas para conversar

1. ¿Es lo mismo registrar muchas operaciones que obtener mayor facturación?
2. ¿Qué resulta más fácil de comprender: 100 filas o un gráfico resumido?
3. ¿Un gráfico bonito siempre representa correctamente la información?

### Ruta del análisis

En la clase anterior llegamos hasta la agrupación. Hoy completaremos la comunicación de resultados:

1. Datos originales.
2. Limpieza.
3. Análisis exploratorio.
4. Agrupación y resumen.
5. Tablas comparativas.
6. Visualización.
7. Interpretación y decisiones.

In [ ]:
# Importación de las herramientas necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.style.use('seaborn-v0_8-whitegrid')

### Dataset de continuidad

La siguiente celda genera y deja limpio el dataset utilizado en la clase anterior. Así todos podrán trabajar sin depender de archivos externos. Cada fila representa una operación de venta.

In [ ]:
rng = np.random.default_rng(2026)
n = 120
productos = np.array(['Cuaderno', 'Bolígrafo', 'Carpeta', 'Regla', 'Mochila', 'Calculadora'])
categorias = {
    'Cuaderno': 'Papelería', 'Bolígrafo': 'Papelería',
    'Carpeta': 'Papelería', 'Regla': 'Útiles',
    'Mochila': 'Accesorios', 'Calculadora': 'Tecnología'
}
precios = {
    'Cuaderno': 18000, 'Bolígrafo': 5000, 'Carpeta': 12000,
    'Regla': 7000, 'Mochila': 95000, 'Calculadora': 65000
}
producto = rng.choice(productos, size=n, p=[0.25, 0.23, 0.16, 0.12, 0.10, 0.14])
df_limpio = pd.DataFrame({
    'fecha': pd.to_datetime('2026-03-01') + pd.to_timedelta(rng.integers(0, 92, size=n), unit='D'),
    'producto': producto,
    'categoria': [categorias[p] for p in producto],
    'cantidad': rng.integers(1, 8, size=n),
    'precio_unitario': [precios[p] for p in producto],
    'sucursal': rng.choice(['Centro', 'Terminal', 'Mercado'], size=n),
    'vendedor': rng.choice(['Ana', 'Carlos', 'Lucía', 'Miguel'], size=n)
})
df_limpio['total_venta'] = df_limpio['cantidad'] * df_limpio['precio_unitario']
df_limpio['mes'] = df_limpio['fecha'].dt.month
df_limpio['nombre_mes'] = df_limpio['mes'].map({3: 'Marzo', 4: 'Abril', 5: 'Mayo'})
df_limpio.head()

### Activación de conocimientos previos

Antes de avanzar, responda usando Pandas:

1. ¿Cuántas filas y columnas tiene el dataset?
2. ¿Cuál es la facturación total?
3. ¿Qué sucursal tiene mayor facturación?
4. ¿Qué diferencia existe entre `groupby()` y un filtro?

In [ ]:
# 1. Filas y columnas del dataset
print(df_limpio.shape)

# 2. Facturación total
print(df_limpio['total_venta'].sum())

# 3. Sucursal con mayor facturación
print(df_limpio.groupby('sucursal')['total_venta'].sum())
print(df_limpio.groupby('sucursal')['total_venta'].sum().idxmax())

# 4. groupby() agrupa y resume (una fila por categoría);
# un filtro selecciona filas sin resumirlas, mantiene el detalle original.


## 2. Tablas dinámicas con `pivot_table()` 

### Concepto principal

Una **tabla dinámica** resume una gran cantidad de registros y permite compararlos mediante filas y columnas.

Su estructura básica es:

```python
pd.pivot_table(
    datos,
    values='columna_numerica',
    index='filas',
    columns='columnas',
    aggfunc='operacion',
    fill_value=0
)
```

| Parámetro | Función |
|---|---|
| `values` | Indica qué valores numéricos se resumirán |
| `index` | Define las categorías que aparecerán en las filas |
| `columns` | Define las categorías que aparecerán en las columnas |
| `aggfunc` | Establece la operación: suma, promedio, conteo, etc. |
| `fill_value` | Reemplaza combinaciones vacías, normalmente por cero |

### Ejemplo 1: facturación por sucursal y categoría

Pregunta: **¿Cuánto facturó cada categoría en cada sucursal?**

In [ ]:
tabla_sucursal_categoria = pd.pivot_table(
    df_limpio,
    values='total_venta',
    index='sucursal',
    columns='categoria',
    aggfunc='sum',
    fill_value=0
)
tabla_sucursal_categoria

### Cómo leer la tabla

- Una **fila** contiene los resultados de una sucursal.
- Una **columna** contiene los resultados de una categoría.
- Cada **celda** es la suma de `total_venta` de esa combinación.
- El cero indica que no hubo registros para la combinación.

No basta con mostrar la tabla: debemos convertir sus números en afirmaciones verificables.

### Ejemplo 2: unidades vendidas por producto y mes


In [ ]:
tabla_producto_mes = pd.pivot_table(
    df_limpio, values='cantidad', index='producto', columns='nombre_mes',
    aggfunc='sum', fill_value=0
).reindex(columns=['Marzo', 'Abril', 'Mayo'])
tabla_producto_mes

### Ejemplo 3: promedio y totales generales

`margins=True` agrega una fila y una columna de totales. Aquí calculamos el promedio de cada operación, no la suma.

In [ ]:
promedio_vendedor_sucursal = pd.pivot_table(
    df_limpio, values='total_venta', index='vendedor', columns='sucursal',
    aggfunc='mean', fill_value=0, margins=True, margins_name='Promedio general'
).round(2)
promedio_vendedor_sucursal

### Ejemplo 4: varias operaciones en una tabla

También podemos solicitar más de una función de resumen.

In [ ]:
tabla_multiple = pd.pivot_table(
    df_limpio, values='total_venta', index='categoria',
    aggfunc=['sum', 'mean', 'count']
).round(2)
tabla_multiple

### `groupby()` frente a `pivot_table()`

| Herramienta | Conviene usarla cuando... |
|---|---|
| `groupby()` | Necesitamos agrupar y producir una serie o tabla lineal |
| `pivot_table()` | Necesitamos comparar dos categorías en filas y columnas |

Ambas pueden obtener resultados equivalentes. La diferencia principal está en la forma de organizar la salida.

## 3. Práctica guiada 1 

Resuelva los siguientes ejercicios. Debajo de cada resultado escriba una oración interpretativa.

1. Cree una tabla con la **facturación total por vendedor y mes**.
2. Cree una tabla con las **unidades vendidas por sucursal y producto**.
3. Calcule el **promedio de venta por categoría y sucursal**.
4. Repita el ejercicio 1 incluyendo totales mediante `margins=True`.
5. Identifique, observando sus tablas, qué vendedor facturó más y qué combinación producto-sucursal vendió más unidades.

In [ ]:
# Ejercicio 1: facturación total por vendedor y mes
tabla_vendedor_mes = pd.pivot_table(
    df_limpio, values='total_venta', index='vendedor', columns='nombre_mes',
    aggfunc='sum', fill_value=0
)
tabla_vendedor_mes

In [ ]:
# Ejercicio 2: unidades vendidas por sucursal y producto
tabla_sucursal_producto = pd.pivot_table(
    df_limpio, values='cantidad', index='sucursal', columns='producto',
    aggfunc='sum', fill_value=0
)
tabla_sucursal_producto

In [ ]:
# Ejercicio 3: promedio de venta por categoría y sucursal
tabla_categoria_sucursal = pd.pivot_table(
    df_limpio, values='total_venta', index='categoria', columns='sucursal',
    aggfunc='mean', fill_value=0
).round(2)
print(tabla_categoria_sucursal)

# Ejercicio 4: ejercicio 1 con totales (margins=True)
tabla_vendedor_mes_totales = pd.pivot_table(
    df_limpio, values='total_venta', index='vendedor', columns='nombre_mes',
    aggfunc='sum', fill_value=0, margins=True, margins_name='Total'
)
tabla_vendedor_mes_totales

### Interpretación de la práctica

- Hallazgo 1: Miguel es el vendedor con mayor facturación total, acumulando Gs. 3.547.000 en el trimestre.
- Hallazgo 2: La combinación Mercado–Carpeta es la que más unidades vendió, con 50 unidades.
- Evidencia numérica utilizada: columna `Total` de la tabla del ejercicio 4 (margins=True) y la celda máxima de la tabla del ejercicio 2.

## 4. Tablas de frecuencia con `pd.crosstab()` 

Una **tabla de frecuencia cruzada** cuenta cuántas veces aparece cada combinación de dos variables categóricas.

Pregunta: **¿Cuántas operaciones registró cada vendedor en cada sucursal?**

In [ ]:
frecuencia_vendedor_sucursal = pd.crosstab(
    df_limpio['vendedor'],
    df_limpio['sucursal'],
    margins=True,
    margins_name='Total'
)
frecuencia_vendedor_sucursal

### Frecuencias porcentuales

Con `normalize='index'` cada fila suma 100 %. Esto ayuda a comparar distribuciones aunque los vendedores tengan distinta cantidad de operaciones.

In [ ]:
porcentaje_vendedor_sucursal = pd.crosstab(
    df_limpio['vendedor'], df_limpio['sucursal'], normalize='index'
) * 100
porcentaje_vendedor_sucursal.round(1)

### Diferencia fundamental

- `crosstab()` responde normalmente **cuántos registros existen**.
- `pivot_table()` responde normalmente **cuánto suman, promedian o representan los valores**.

Una persona puede aparecer en muchas operaciones pequeñas y, aun así, no ser quien más facturó.

### Práctica guiada 2

1. Construya una tabla de frecuencias entre `producto` y `sucursal`.
2. Agregue los totales.
3. Obtenga porcentajes por fila.
4. Explique por qué esta tabla no muestra directamente la facturación.

In [ ]:
# 1 y 2: tabla de frecuencias producto x sucursal, con totales
tabla_producto_sucursal = pd.crosstab(
    df_limpio['producto'], df_limpio['sucursal'],
    margins=True, margins_name='Total'
)
print(tabla_producto_sucursal)

# 3: porcentajes por fila
porcentaje_producto_sucursal = pd.crosstab(
    df_limpio['producto'], df_limpio['sucursal'], normalize='index'
) * 100
print(porcentaje_producto_sucursal.round(1))

# 4. No muestra la facturación porque crosstab() solo cuenta operaciones (filas),
# sin mirar la columna total_venta. Un producto puede tener muchas ventas
# pequeñas y aun así facturar menos que otro con pocas ventas caras.

## Pausa activa y revisión 

Antes de continuar, complete verbalmente:

- Una tabla dinámica sirve para...
- El parámetro `aggfunc` sirve para...
- `crosstab()` es útil cuando necesitamos...
- Contar operaciones no es igual que sumar facturación porque...

## 5. Visualización de datos 
### Elegir el gráfico correcto

| Pregunta | Gráfico recomendado |
|---|---|
| ¿Qué categoría vende más? | Barras |
| ¿Cómo cambia la venta con el tiempo? | Líneas |
| ¿Qué proporción representa cada categoría? | Circular, con pocas categorías |
| ¿Cómo se comparan categorías entre sucursales? | Barras agrupadas o apiladas |

Un buen gráfico debe tener título claro, nombres de ejes, unidades, escala legible y colores que no confundan.

### Ejemplo 1: gráfico de barras para comparar categorías


In [ ]:
ventas_categoria = df_limpio.groupby('categoria')['total_venta'].sum().sort_values()
ax = ventas_categoria.plot(kind='barh', color='#3478BF', figsize=(9, 5))
ax.set_title('Facturación total por categoría')
ax.set_xlabel('Facturación en guaraníes')
ax.set_ylabel('Categoría')
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'Gs. {x:,.0f}'))
plt.tight_layout()
plt.show()

### Ejemplo 2: gráfico de líneas para observar cambios

Para una secuencia temporal debemos respetar el orden cronológico.

In [ ]:
orden_meses = ['Marzo', 'Abril', 'Mayo']
ventas_mes = df_limpio.groupby('nombre_mes')['total_venta'].sum().reindex(orden_meses)
ax = ventas_mes.plot(kind='line', marker='o', linewidth=3, color='#2E8B57', figsize=(9, 5))
ax.set_title('Evolución mensual de la facturación')
ax.set_xlabel('Mes')
ax.set_ylabel('Facturación en guaraníes')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'Gs. {x:,.0f}'))
plt.tight_layout()
plt.show()

### Ejemplo 3: barras agrupadas a partir de una tabla dinámica


In [ ]:
ax = tabla_sucursal_categoria.plot(kind='bar', figsize=(11, 6))
ax.set_title('Facturación por sucursal y categoría')
ax.set_xlabel('Sucursal')
ax.set_ylabel('Facturación en guaraníes')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Categoría')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'Gs. {x:,.0f}'))
plt.tight_layout()
plt.show()

### Ejemplo 4: gráfico circular y uso responsable

El gráfico circular funciona mejor con pocas categorías. Si hay demasiadas porciones, las comparaciones se vuelven difíciles.

In [ ]:
ventas_categoria.plot(
    kind='pie', autopct='%1.1f%%', startangle=90, figsize=(7, 7),
    ylabel='', title='Participación de cada categoría en la facturación'
)
plt.tight_layout()
plt.show()

### Errores frecuentes que debemos evitar

- usar un gráfico de líneas para categorías sin orden temporal;
- omitir títulos o unidades;
- recargar el gráfico con demasiados colores;
- utilizar un gráfico circular con demasiadas porciones;
- afirmar una causa cuando los datos solo muestran una diferencia;
- modificar la escala para exagerar cambios pequeños.

## 6. Práctica guiada 3: gráficos e interpretación 

Realice las siguientes actividades:

1. Calcule la facturación por producto y represéntela con barras horizontales.
2. Calcule la facturación mensual de cada vendedor y represéntela mediante líneas.
3. Represente las unidades vendidas por sucursal y producto con barras agrupadas.
4. Personalice títulos, nombres de ejes, colores y tamaño.
5. Debajo de cada gráfico escriba un hallazgo que incluya evidencia numérica.

In [ ]:
# Ejercicio gráfico 1: facturación por producto
ventas_producto = df_limpio.groupby('producto')['total_venta'].sum().sort_values()
ax = ventas_producto.plot(kind='barh', color='#3478BF', figsize=(9, 5))
ax.set_title('Facturación total por producto')
ax.set_xlabel('Facturación en guaraníes')
ax.set_ylabel('Producto')
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'Gs. {x:,.0f}'))
plt.tight_layout()
plt.show()

**Interpretación del gráfico 1:**  
Mochila es el producto que más factura, con Gs. 3.800.000, seguido de cerca por Calculadora (Gs. 3.445.000). Ambos superan ampliamente a Regla, el de menor facturación (Gs. 413.000), lo que sugiere que los productos de mayor precio unitario concentran gran parte de los ingresos aunque se vendan menos unidades.

In [ ]:
# Ejercicio gráfico 2: evolución mensual por vendedor
orden_meses = ['Marzo', 'Abril', 'Mayo']
tabla_vendedor_mes_grafico = (
    df_limpio.groupby(['nombre_mes', 'vendedor'])['total_venta']
    .sum().unstack().reindex(orden_meses)
)
ax = tabla_vendedor_mes_grafico.plot(kind='line', marker='o', figsize=(10, 5))
ax.set_title('Evolución mensual de la facturación por vendedor')
ax.set_xlabel('Mes')
ax.set_ylabel('Facturación en guaraníes')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'Gs. {x:,.0f}'))
ax.legend(title='Vendedor')
plt.tight_layout()
plt.show()

**Interpretación del gráfico 2:**  
Miguel lidera en marzo con Gs. 2.005.000, pero cae fuertemente en mayo (Gs. 536.000), mientras que Ana pasa de ser la más baja en marzo (Gs. 556.000) a la más alta en abril (Gs. 1.857.000). Esto muestra que el rendimiento de cada vendedor varía bastante mes a mes, sin un líder constante durante todo el trimestre.

In [ ]:
# Ejercicio gráfico 3: unidades por sucursal y producto
tabla_sucursal_producto_grafico = pd.pivot_table(
    df_limpio, values='cantidad', index='sucursal', columns='producto',
    aggfunc='sum', fill_value=0
)
ax = tabla_sucursal_producto_grafico.plot(kind='bar', figsize=(11, 6))
ax.set_title('Unidades vendidas por sucursal y producto')
ax.set_xlabel('Sucursal')
ax.set_ylabel('Unidades vendidas')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Producto')
plt.tight_layout()
plt.show()

**Interpretación del gráfico 3:**  
Mercado es la sucursal con más unidades de Carpeta vendidas (50), mientras que Terminal vende notablemente menos Bolígrafo (20) que Centro (43) y Mercado (45). Esto indica que la mezcla de productos vendida no es homogénea entre sucursales, algo que una tabla de frecuencias por sí sola no deja ver tan claramente.

## 7. Cómo interpretar correctamente 

Una conclusión sólida contiene tres elementos:

1. **Hallazgo:** qué se observa.
2. **Evidencia:** número o comparación que lo demuestra.
3. **Implicación:** por qué puede ser importante.

### Modelo

> La categoría Tecnología presenta la mayor facturación, con Gs. X. Esto indica que, aunque no necesariamente tenga más operaciones, sus ventas aportan una parte importante de los ingresos.

Observe que no afirmamos la causa. Para decir *por qué* ocurrió necesitaríamos otros datos o una investigación adicional.

## 8. Desafío integrador: miniinforme gerencial 

La administración solicita un informe breve para orientar una campaña comercial. Prepare lo siguiente:

1. Una tabla dinámica de facturación por sucursal y categoría.
2. Una tabla de frecuencia de operaciones por vendedor y sucursal.
3. Un gráfico de barras que permita comparar productos.
4. Un gráfico de líneas que muestre la evolución mensual.
5. Tres conclusiones respaldadas por números.
6. Dos recomendaciones concretas para la librería.

### Criterios de logro

| Criterio | Puntaje |
|---|---:|
| Tablas correctas y legibles | 3 |
| Gráficos adecuados y personalizados | 3 |
| Conclusiones con evidencia | 2 |
| Recomendaciones coherentes | 2 |
| **Total** | **10** |

In [ ]:
# 1. Tabla dinámica: facturación por sucursal y categoría
tabla_sucursal_categoria_final = pd.pivot_table(
    df_limpio, values='total_venta', index='sucursal', columns='categoria',
    aggfunc='sum', fill_value=0
)
print(tabla_sucursal_categoria_final)

# 2. Tabla de frecuencia: operaciones por vendedor y sucursal
frecuencia_vendedor_sucursal_final = pd.crosstab(
    df_limpio['vendedor'], df_limpio['sucursal'],
    margins=True, margins_name='Total'
)
print(frecuencia_vendedor_sucursal_final)

# 3. Gráfico de barras: comparación de productos
ventas_producto_final = df_limpio.groupby('producto')['total_venta'].sum().sort_values()
ax = ventas_producto_final.plot(kind='barh', color='#2E8B57', figsize=(9, 5))
ax.set_title('Facturación por producto')
ax.set_xlabel('Facturación en guaraníes')
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'Gs. {x:,.0f}'))
plt.tight_layout()
plt.show()

# 4. Gráfico de líneas: evolución mensual total
ventas_mes_final = df_limpio.groupby('nombre_mes')['total_venta'].sum().reindex(orden_meses)
ax = ventas_mes_final.plot(kind='line', marker='o', linewidth=3, color='#B22222', figsize=(9, 5))
ax.set_title('Evolución mensual de la facturación')
ax.set_xlabel('Mes')
ax.set_ylabel('Facturación en guaraníes')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'Gs. {x:,.0f}'))
plt.tight_layout()
plt.show()

### Miniinforme

**Conclusión 1:**  
La sucursal Centro es la que más factura en total (Gs. 4.118.000), impulsada principalmente por la categoría Accesorios (Gs. 1.710.000).

**Conclusión 2:**  
La categoría Accesorios (representada por Mochila) es la de mayor facturación general con Gs. 3.800.000, a pesar de no ser la más vendida en unidades, lo que indica un alto valor por operación.

**Conclusión 3:**  
Carlos registra la mayor cantidad de operaciones (33), pero Miguel factura más en total (Gs. 3.547.000), lo que confirma que más operaciones no equivalen automáticamente a mayor facturación.

**Recomendación 1:**  
Reforzar el stock y la promoción de Mochila y Calculadora en la sucursal Centro, ya que son los productos de mayor ticket y esa sucursal concentra la mayor facturación.

**Recomendación 2:**  
Analizar por qué Terminal tiene menor facturación en Papelería en comparación con las otras sucursales, para evaluar si conviene ajustar precios o reforzar la oferta de ese rubro allí.

## 9. Cierre y metacognición — 5 minutos

Complete las siguientes frases:

- Hoy aprendí que una tabla dinámica...
- El gráfico más útil para comparar categorías es...
- Antes de formular una conclusión debo...
- Una dificultad que todavía necesito practicar es...

### Lista de verificación

- [ ] Construí una tabla con `pivot_table()`.
- [ ] Utilicé correctamente `aggfunc`.
- [ ] Construí una tabla con `crosstab()`.
- [ ] Elegí el gráfico según la pregunta.
- [ ] Incluí título, ejes y unidades.
- [ ] Redacté conclusiones con evidencia numérica.
- [ ] Evité atribuir causas que los datos no demuestran.